## Boundary conditions and a first linear elasticity solve

Builds on [06_multipatch_stiffness](06_multipatch_stiffness.ipynb). `future` can now
assemble the stiffness matrix `K`. This notebook adds the two missing pieces for a full
linear elasticity computation: Dirichlet (displacement) boundary conditions, and a
distributed (Neumann) load assembled into a right-hand side `F`.

`future` deliberately stops at producing `K`, `F`, and the list of fixed dofs -- it
provides *bricks*, not a solver. Reducing the system (eliminating the fixed dofs) and
solving it is a handful of lines of `scipy`, shown at the end of this notebook, not new
`future` code.

Two new building blocks make this possible:
- [`Patch.boundary_control_points(direction, side, span_min=-1, span_max=-1)`](../../docs/source/future/api.rst)
  selects control points on a patch edge (or a span sub-range of it) -- combine with
  `dof_manager.get_global_dof_indices(...)` to get the global dofs to fix.
- [`PatchIntegrator.integrate_boundary_load()`](../../docs/source/future/api.rst)/`assemble_boundary_load()`
  integrate a [`Traction`](../../docs/source/future/api.rst) (force per unit length) over an edge or a
  span sub-range of it, into a load vector the same size as `K`.

In [ ]:
import numpy as np
import scipy.sparse as sp
from scipy.sparse.linalg import spsolve
import matplotlib.pyplot as plt

from yeti_iga.future.bspline import (BSpline, BSplineSurface, ControlPointManager,
    Patch, GlobalDOFManager, PatchDOFManager, PatchAssembly, IGABasis1D,
    PatchIntegrator, Material, PlaneStress, ConstantTraction, BoundaryLoadSpec)
from yeti_iga.future.plotting import plot_patches_2d, plot_dirichlet_bc, plot_distributed_load

### The geometry: a cantilever plate

Same 6x1 rectangle, degree 2, two elements as in
[06_multipatch_stiffness](06_multipatch_stiffness.ipynb) -- a short cantilever, clamped
on its left edge (x=0), with a shear load on its right edge (x=6).

In [ ]:
mgr = ControlPointManager(dim=2)
for y in (0.0, 0.5, 1.0):
    for x in (0.0, 1.5, 3.0, 4.5, 6.0):
        mgr.add_point([x, y])

su = BSpline(2, np.array([0., 0., 0., 0.5, 0.5, 1., 1., 1.]))
sv = BSpline(2, np.array([0., 0., 0., 1., 1., 1.]))
mapping = list(range(15))

dof_manager = GlobalDOFManager([2] * mgr.n_points)
patch_dof_manager = PatchDOFManager(2, mapping, dof_manager)
patch = Patch(BSplineSurface(su, sv), mgr, mapping, [5, 3], patch_dof_manager)

basis_u = IGABasis1D.build(su, 3)
basis_v = IGABasis1D.build(sv, 3)
material = PlaneStress(Material(E=210000, nu=0.3))

integrator = PatchIntegrator(patch, basis_u, basis_v, material)
K = integrator.integrate_stiffness()
print('K shape:', K.shape)

### Dirichlet boundary condition: clamping the left edge

`boundary_control_points(direction=0, side=0)` selects the control points on the x=0
edge (fixing direction 0 at its first local index). Both displacement components are
fixed to 0 at every one of them -- a fully clamped edge.

In [ ]:
left_edge_cps = patch.boundary_control_points(0, 0)
print('Left edge control points (local positions):', left_edge_cps)

fixed_dofs = []
for cp in left_edge_cps:
    fixed_dofs.extend(patch.dof_manager.get_global_dof_indices(cp))
fixed_dofs = np.array(sorted(fixed_dofs))
print('Fixed dofs:', fixed_dofs)

### Neumann boundary condition: a shear load on the right edge

`integrate_boundary_load(direction=0, side=1, traction)` integrates a constant traction
`(0, -p)` over the x=6 edge -- physically a downward shear force, like a tip load on the
cantilever. The resultant (sum of the y-components) must equal `-p * L` with `L = 1.0`
the edge length, a useful sanity check independent of the IGA machinery itself.

In [ ]:
p = 1000.0  # N/m
traction = ConstantTraction(np.array([0.0, -p]))
F = integrator.integrate_boundary_load(0, 1, traction)

print('F shape:', F.shape)
print('resultant Fy:', F[1::2].sum(), '(expected:', -p * 1.0, ')')
assert np.isclose(F[1::2].sum(), -p * 1.0)

### Visualizing the boundary conditions

`plot_dirichlet_bc()` and `plot_distributed_load()` draw on top of the axes returned by
`plot_patches_2d()`: a red square at every fully-clamped control point, and arrows along
the loaded edge showing the traction `(0, -p)` sampled from the same `Traction` object
used for the actual integration.

In [ ]:
fig, ax = plt.subplots()
plot_patches_2d(patch, show_control_points=True, ax=ax,
                title='Cantilever: clamped left edge, sheared right edge')
plot_dirichlet_bc(ax, patch, left_edge_cps, dofs=(0, 1))
plot_distributed_load(ax, patch, 0, 1, traction)
plt.show()

### Reducing and solving the system (scipy, not `future`)

This is the part `future` does *not* provide: eliminate the fixed dofs, solve the
reduced system for the free displacements, and put the (zero) prescribed values back at
the fixed dofs. A handful of lines, since `fixed_dofs`/`F`/`K` are already in exactly
the form `scipy` needs.

In [ ]:
all_dofs = np.arange(K.shape[0])
free_dofs = np.setdiff1d(all_dofs, fixed_dofs)

K_free = K.tocsc()[free_dofs, :][:, free_dofs]
F_free = F[free_dofs]

d = np.zeros(K.shape[0])
d[free_dofs] = spsolve(K_free, F_free)

print('max |displacement|:', np.abs(d).max())
print('displacement at the clamped edge (must be 0):', d[fixed_dofs])

# The tip (x=6) must move down (negative y) under the downward shear load.
right_edge_cps = patch.boundary_control_points(0, 1)
right_edge_uy = [d[patch.dof_manager.get_global_dof_indices(cp)[1]] for cp in right_edge_cps]
print('uy at the loaded edge:', right_edge_uy)
assert np.allclose(d[fixed_dofs], 0.0)
assert all(uy < 0 for uy in right_edge_uy)

In [ ]:
mgr_deformed = ControlPointManager(dim=2)
scale = 0.2 * 6.0 / max(np.abs(d).max(), 1.e-12)
for cp in range(patch.n_cp):
    dofs = patch.dof_manager.get_global_dof_indices(cp)
    mgr_deformed.add_point((patch.control_point(cp) + scale * d[dofs]).tolist())

patch_deformed = Patch(BSplineSurface(su, sv), mgr_deformed, mapping, [5, 3])

ax = plot_patches_2d([patch, patch_deformed], show_control_points=True,
                     title='Cantilever plate: undeformed vs. deformed shape')
ax.plot([], [], color=plt.cm.tab10(0), label='undeformed')
ax.plot([], [], color=plt.cm.tab10(1), label=f'deformed (x{scale:.1f})')
ax.legend()
plt.show()

### Multipatch: the same case, split in two

Same domain split as [06_multipatch_stiffness](06_multipatch_stiffness.ipynb) -- the BC
and load only touch ONE of the two patches each (the left patch's x=0 edge for the
clamp, the right patch's x=6 edge for the load), yet `assemble_boundary_load()` already
sizes its result to the assembly-wide dof count, so it composes directly with
`assemble_stiffness()` exactly like the single-patch case above.

In [ ]:
mgr_multi = ControlPointManager(dim=2)
for y in (0.0, 0.5, 1.0):
    for x in (0.0, 1.5, 3.0, 4.5, 6.0):
        mgr_multi.add_point([x, y])

dofs_per_control_point = [2] * mgr_multi.n_points
global_dof_manager = GlobalDOFManager(dofs_per_control_point)

su_left = BSpline(2, np.array([0., 0., 0., 1., 1., 1.]))
sv_left = BSpline(2, np.array([0., 0., 0., 1., 1., 1.]))
mapping_left = [0, 1, 2, 5, 6, 7, 10, 11, 12]
dof_manager_left = PatchDOFManager(2, mapping_left, global_dof_manager)
patch_left = Patch(BSplineSurface(su_left, sv_left), mgr_multi, mapping_left, [3, 3],
                    dof_manager_left)

su_right = BSpline(2, np.array([0., 0., 0., 1., 1., 1.]))
sv_right = BSpline(2, np.array([0., 0., 0., 1., 1., 1.]))
mapping_right = [2, 3, 4, 7, 8, 9, 12, 13, 14]
dof_manager_right = PatchDOFManager(2, mapping_right, global_dof_manager)
patch_right = Patch(BSplineSurface(su_right, sv_right), mgr_multi, mapping_right, [3, 3],
                     dof_manager_right)

assembly = PatchAssembly()
assembly.add_patch(patch_left)
assembly.add_patch(patch_right)
assembly.detect_shared_control_points()

materials = [material, material]
K_multi = PatchIntegrator.assemble_stiffness(assembly, materials)

spec = BoundaryLoadSpec(patch_index=1, direction=0, side=1, traction=traction)
F_multi = PatchIntegrator.assemble_boundary_load(assembly, [spec])

left_edge_cps_multi = patch_left.boundary_control_points(0, 0)
fixed_dofs_multi = []
for cp in left_edge_cps_multi:
    fixed_dofs_multi.extend(patch_left.dof_manager.get_global_dof_indices(cp))
fixed_dofs_multi = np.array(sorted(fixed_dofs_multi))

free_dofs_multi = np.setdiff1d(np.arange(K_multi.shape[0]), fixed_dofs_multi)
d_multi = np.zeros(K_multi.shape[0])
d_multi[free_dofs_multi] = spsolve(
    K_multi.tocsc()[free_dofs_multi, :][:, free_dofs_multi], F_multi[free_dofs_multi])

print('max abs difference vs. the single-patch solution:', np.abs(d_multi - d).max())
assert np.allclose(d_multi, d, rtol=1.e-8, atol=1.e-8)
print('multipatch solution matches the single-patch reference: OK')